# Amazon Deforestation: Sentinel-1 + Sentinel-2 Late-Fusion SegNet

**Separate Kaggle experiment.** Sentinel-1 SAR and Sentinel-2 optical are two sensor modalities, not before/after dates. This notebook downloads only `akhilchibber/deforestation-detection-dataset`; it does not use the Forest-Change dataset or the existing project models.

Run the cells in order in Google Colab. The notebook checks the actual archive, TIFF contents, bands, label values, geospatial alignment and spatial splits before training. It stops if those checks fail; it does not invent labels or silently resample rasters.

## 1. Select a free Colab GPU, add the Kaggle secret, then mount Drive

In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available free GPU). The notebook does not enable a paid runtime. In the left **Secrets** panel create `KAGGLE_API_TOKEN` with a Kaggle API token and grant this notebook access. Never paste a token into a code cell.

In [ ]:
%pip -q install kaggle rasterio matplotlib pandas scikit-learn

import os, sys, json, math, time, random, hashlib, shutil, zipfile, subprocess, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Deforestation_Research/Kaggle_Amazon')
RAW_DIR = DRIVE_ROOT / 'raw'
EXTRACT_DIR = RAW_DIR / 'extracted'
META_DIR = DRIVE_ROOT / 'metadata'
PATCH_DIR = DRIVE_ROOT / 'processed'
MODEL_DIR = DRIVE_ROOT / 'models'
RESULT_DIR = DRIVE_ROOT / 'results'
VIS_DIR = DRIVE_ROOT / 'visualizations'
CODE_DIR = DRIVE_ROOT / 'experiment_code'
for folder in (RAW_DIR, EXTRACT_DIR, META_DIR, PATCH_DIR, MODEL_DIR, RESULT_DIR, VIS_DIR, CODE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB')
else:
    print('No GPU was assigned. Select a free GPU runtime before running the training cell.')

usage = shutil.disk_usage(str(DRIVE_ROOT))
print(f'Mounted Drive filesystem reports {usage.free / 1024**3:.1f} GiB free. Verify your actual Google Drive quota in Drive; FUSE free-space values can be approximate.')
print('Permanent dataset root:', DRIVE_ROOT)

## 2. Save the experiment modules to Drive

This notebook embeds the same source as the separately delivered `model.py` and `dataset.py`, so it remains runnable after the notebook is uploaded by itself. The copies are saved in Drive and imported below.

In [ ]:
MODEL_SOURCE = '"""Two-stream late-fusion SegNet for Sentinel-1 SAR + Sentinel-2 optical inputs.\n\nSentinel-1 and Sentinel-2 are separate sensor modalities. This model does not\ninterpret them as temporal observations.\n"""\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, in_channels: int, out_channels: int, layers: int = 2):\n        modules = []\n        for index in range(layers):\n            source = in_channels if index == 0 else out_channels\n            modules.extend([\n                nn.Conv2d(source, out_channels, kernel_size=3, padding=1, bias=False),\n                nn.BatchNorm2d(out_channels),\n                nn.ReLU(inplace=True),\n            ])\n        super().__init__(*modules)\n\n\nclass SegNetEncoder(nn.Module):\n    """VGG-style encoder that retains max-pool indices for SegNet decoding."""\n    def __init__(self, in_channels: int, channels: tuple[int, ...]):\n        super().__init__()\n        depths = (2, 2, 3, 3, 3)[:len(channels)]\n        blocks, current = [], in_channels\n        for width, depth in zip(channels, depths):\n            blocks.append(ConvBlock(current, width, depth))\n            current = width\n        self.blocks = nn.ModuleList(blocks)\n        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)\n\n    def forward(self, x: torch.Tensor):\n        indices, pre_pool_shapes = [], []\n        for block in self.blocks:\n            x = block(x)\n            pre_pool_shapes.append(x.size())\n            x, index = self.pool(x)\n            indices.append(index)\n        return x, indices, pre_pool_shapes\n\n\nclass SegNetDecoder(nn.Module):\n    def __init__(self, channels: tuple[int, ...]):\n        super().__init__()\n        self.unpool = nn.MaxUnpool2d(kernel_size=2, stride=2)\n        blocks = []\n        for level in range(len(channels) - 1, -1, -1):\n            source = channels[level]\n            target = channels[level - 1] if level > 0 else channels[0]\n            depth = 3 if level >= 3 else 2\n            blocks.append(ConvBlock(source, target, depth))\n        self.blocks = nn.ModuleList(blocks)\n\n    def forward(self, x: torch.Tensor, indices, pre_pool_shapes):\n        for block, level in zip(self.blocks, range(len(indices) - 1, -1, -1)):\n            x = self.unpool(x, indices[level], output_size=pre_pool_shapes[level])\n            x = block(x)\n        return x\n\n\nclass LateFusionSegNet(nn.Module):\n    """Independent modality encoders/decoders fused at decoded feature level.\n\n    Args:\n        s1_channels: Sentinel-1 band count (the source research notebook uses 2).\n        s2_channels: Sentinel-2 band count (the source research notebook uses 4).\n        base_channels: Width of the first encoder stage. 32 is sized for Colab GPUs.\n    Returns one raw binary logit per pixel; use BCEWithLogits/Dice during training.\n    """\n    def __init__(self, s1_channels: int = 2, s2_channels: int = 4,\n                 base_channels: int = 32):\n        super().__init__()\n        channels = (base_channels, base_channels * 2, base_channels * 4,\n                    base_channels * 8, base_channels * 8)\n        self.s1_encoder = SegNetEncoder(s1_channels, channels)\n        self.s1_decoder = SegNetDecoder(channels)\n        self.s2_encoder = SegNetEncoder(s2_channels, channels)\n        self.s2_decoder = SegNetDecoder(channels)\n        self.fusion = nn.Sequential(\n            nn.Conv2d(channels[0] * 2, channels[0], kernel_size=1, bias=False),\n            nn.BatchNorm2d(channels[0]),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(channels[0], 1, kernel_size=1),\n        )\n        self._initialize_weights()\n\n    def _initialize_weights(self):\n        for module in self.modules():\n            if isinstance(module, nn.Conv2d):\n                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")\n                if module.bias is not None:\n                    nn.init.zeros_(module.bias)\n            elif isinstance(module, nn.BatchNorm2d):\n                nn.init.ones_(module.weight)\n                nn.init.zeros_(module.bias)\n\n    def forward(self, sentinel1: torch.Tensor, sentinel2: torch.Tensor) -> torch.Tensor:\n        if sentinel1.shape[-2:] != sentinel2.shape[-2:]:\n            raise ValueError("Sentinel-1 and Sentinel-2 tensors must share spatial dimensions")\n        h, w = sentinel1.shape[-2:]\n        if h % 32 or w % 32:\n            raise ValueError("Input height and width must be divisible by 32 (e.g. 256x256)")\n        z1, i1, s1 = self.s1_encoder(sentinel1)\n        z2, i2, s2 = self.s2_encoder(sentinel2)\n        d1 = self.s1_decoder(z1, i1, s1)\n        d2 = self.s2_decoder(z2, i2, s2)\n        return self.fusion(torch.cat((d2, d1), dim=1))\n'
DATASET_SOURCE = '"""GeoTIFF pairing, spatial windows, normalization, and PyTorch data loading."""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Iterable\nimport random\n\nimport numpy as np\nimport rasterio\nfrom rasterio.windows import Window\nimport torch\nfrom torch.utils.data import Dataset\n\n\n@dataclass(frozen=True)\nclass PatchRecord:\n    tile_id: str\n    split: str\n    s1_path: str\n    s2_path: str\n    mask_path: str\n    row: int\n    col: int\n    patch_size: int\n    valid_fraction: float\n    positive_fraction: float\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef canonical_tile_id(path: str | Path) -> str:\n    """Get matching grid id from names such as RASTER_12.tif or MASK_RASTER_12.tif."""\n    import re\n    stem = Path(path).stem\n    match = re.search(r"raster[_-]?(\\d+)", stem, flags=re.IGNORECASE)\n    return f"RASTER_{int(match.group(1))}" if match else stem.casefold()\n\n\ndef identify_modality(path: str | Path) -> tuple[str | None, str]:\n    parts = [part.casefold().replace("-", "_").replace(" ", "_") for part in Path(path).parts]\n    joined = "_".join(parts)\n    if any("mask" in part or "training_label" in part or "prodes" in part for part in parts):\n        return "mask", "shared"\n    if "sentinel1" in joined or "sentinel_1" in joined or "sar" in joined:\n        modality = "s1"\n    elif "sentinel2" in joined or "sentinel_2" in joined or "optical" in joined:\n        modality = "s2"\n    else:\n        return None, "unknown"\n    if "cloud_free" in joined or "cloudfree" in joined:\n        domain = "cloud_free"\n    elif "cloudy" in joined or "cloud" in joined:\n        domain = "cloudy"\n    else:\n        domain = "unspecified"\n    return modality, domain\n\n\ndef raster_signature(path: str | Path) -> dict:\n    with rasterio.open(path) as src:\n        return {\n            "path": str(path), "width": src.width, "height": src.height,\n            "count": src.count, "dtype": ",".join(src.dtypes),\n            "crs": src.crs.to_string() if src.crs else None,\n            "transform": tuple(src.transform)[:6], "bounds": tuple(src.bounds),\n            "nodata": src.nodata, "descriptions": src.descriptions,\n            "block_shapes": src.block_shapes,\n        }\n\n\ndef validate_pair_alignment(s1_path: str | Path, s2_path: str | Path,\n                            mask_path: str | Path) -> tuple[bool, str]:\n    signatures = [raster_signature(p) for p in (s1_path, s2_path, mask_path)]\n    base = signatures[0]\n    for other in signatures[1:]:\n        if (other["width"], other["height"]) != (base["width"], base["height"]):\n            return False, "different raster dimensions"\n        if other["crs"] != base["crs"]:\n            return False, "different or missing CRS"\n        if not np.allclose(other["transform"], base["transform"], rtol=0, atol=1e-8):\n            return False, "different affine transform/pixel alignment"\n    return True, "same dimensions, CRS, and affine transform"\n\n\ndef assign_spatial_splits(samples: list[dict], seed: int = 42,\n                          ratios: dict[str, float] | None = None) -> tuple[dict[str, str], dict[str, str]]:\n    """Assign whole adjacent 2x2 raster blocks to splits using georeferenced centers.\n\n    This intentionally groups nearby grid tiles before splitting. With only four\n    spatial blocks, the realized ratio may be 50/25/25 rather than 70/15/15.\n    Returns tile_id -> split and tile_id -> geographic block id.\n    """\n    if ratios is None:\n        ratios = {"train": 0.70, "val": 0.15, "test": 0.15}\n    if len(samples) < 3:\n        raise ValueError("At least three aligned cloud-free tiles are needed for spatial train/val/test splits")\n    crss = {sample["crs"] for sample in samples}\n    if len(crss) != 1 or None in crss:\n        raise ValueError("Spatial splitting requires one shared, defined CRS")\n    widths = [sample["bounds"][2] - sample["bounds"][0] for sample in samples]\n    heights = [sample["bounds"][3] - sample["bounds"][1] for sample in samples]\n    x_step = float(np.median(widths)) * 2.0\n    y_step = float(np.median(heights)) * 2.0\n    min_x = min(sample["bounds"][0] for sample in samples)\n    min_y = min(sample["bounds"][1] for sample in samples)\n    max_y = max(sample["bounds"][3] for sample in samples)\n    blocks = {}\n    for sample in samples:\n        xmin, ymin, xmax, ymax = sample["bounds"]\n        center_x, center_y = (xmin + xmax) / 2, (ymin + ymax) / 2\n        col = int(np.floor((center_x - min_x) / max(x_step, 1e-9)))\n        row = int(np.floor((max_y - center_y) / max(y_step, 1e-9)))\n        block_id = f"block_r{row}_c{col}"\n        blocks[sample["tile_id"]] = block_id\n    group_to_tiles: dict[str, list[str]] = {}\n    for tile, block in blocks.items():\n        group_to_tiles.setdefault(block, []).append(tile)\n    if len(group_to_tiles) < 3:\n        raise ValueError(f"Only {len(group_to_tiles)} spatial blocks found; cannot form three independent splits")\n    rng = random.Random(seed)\n    groups = list(group_to_tiles)\n    rng.shuffle(groups)\n    groups.sort(key=lambda group: len(group_to_tiles[group]), reverse=True)\n    targets = {name: ratios[name] * len(samples) for name in ("train", "val", "test")}\n    counts = {name: 0 for name in targets}\n    group_split = {}\n    for group in groups:\n        destination = min(targets, key=lambda name: counts[name] / max(targets[name], 1e-9))\n        group_split[group] = destination\n        counts[destination] += len(group_to_tiles[group])\n    tile_splits = {tile: group_split[block] for tile, block in blocks.items()}\n    return tile_splits, blocks\n\n\ndef _read_valid(path: str | Path, window: Window) -> tuple[np.ndarray, np.ndarray]:\n    with rasterio.open(path) as src:\n        values = src.read(window=window).astype(np.float32)\n        band_valid = src.read_masks(window=window) > 0\n        valid = np.all(band_valid, axis=0) & np.all(np.isfinite(values), axis=0)\n        return values, valid\n\n\ndef _read_label(path: str | Path, window: Window, positive_value: int,\n                background_value: int, ignore_values: Iterable[int]) -> tuple[np.ndarray, np.ndarray]:\n    with rasterio.open(path) as src:\n        raw = src.read(1, window=window)\n        valid = np.isin(raw, [positive_value, background_value])\n        if src.nodata is not None:\n            valid &= raw != src.nodata\n        for value in ignore_values:\n            valid &= raw != value\n        target = raw == positive_value\n        return target.astype(np.float32), valid\n\n\ndef build_patch_records(samples: list[dict], tile_splits: dict[str, str],\n                        patch_size: int = 256, stride: int = 256,\n                        positive_value: int = 1, background_value: int = 0,\n                        ignore_values: Iterable[int] = (),\n                        min_valid_fraction: float = 0.90) -> list[PatchRecord]:\n    """Index aligned non-overlapping windows; never split one source tile across sets."""\n    records = []\n    for sample in samples:\n        tile = sample["tile_id"]\n        split = tile_splits[tile]\n        with rasterio.open(sample["mask_path"]) as mask_src:\n            height, width = mask_src.height, mask_src.width\n        for row in range(0, height - patch_size + 1, stride):\n            for col in range(0, width - patch_size + 1, stride):\n                window = Window(col, row, patch_size, patch_size)\n                s1, valid1 = _read_valid(sample["s1_path"], window)\n                s2, valid2 = _read_valid(sample["s2_path"], window)\n                target, valid_mask = _read_label(sample["mask_path"], window,\n                                                 positive_value, background_value, ignore_values)\n                valid = valid1 & valid2 & valid_mask\n                fraction = float(valid.mean())\n                if fraction < min_valid_fraction:\n                    continue\n                pos_fraction = float(target[valid].mean()) if valid.any() else 0.0\n                records.append(PatchRecord(tile, split, sample["s1_path"], sample["s2_path"],\n                                           sample["mask_path"], row, col, patch_size,\n                                           fraction, pos_fraction))\n    return records\n\n\ndef fit_robust_ranges(records: list[PatchRecord], percentiles=(1.0, 99.0),\n                      max_windows: int = 800, points_per_window: int = 512,\n                      seed: int = 42, positive_value: int = 1,\n                      background_value: int = 0,\n                      ignore_values: Iterable[int] = ()) -> dict[str, dict[str, list[float]]]:\n    """Fit per-band percentile ranges on TRAIN windows only."""\n    train = [record for record in records if record.split == "train"]\n    if not train:\n        raise ValueError("No training windows available to fit normalization")\n    rng = np.random.default_rng(seed)\n    chosen = rng.choice(len(train), size=min(max_windows, len(train)), replace=False)\n    buckets = {"s1": None, "s2": None}\n    for index in chosen:\n        record = train[int(index)]\n        window = Window(record.col, record.row, record.patch_size, record.patch_size)\n        arrays = {}\n        valid = np.ones((record.patch_size, record.patch_size), dtype=bool)\n        for modality, path in (("s1", record.s1_path), ("s2", record.s2_path)):\n            arrays[modality], band_valid = _read_valid(path, window)\n            valid &= band_valid\n        _, mask_valid = _read_label(record.mask_path, window, positive_value,\n                                    background_value, ignore_values)\n        valid &= mask_valid\n        for modality, array in arrays.items():\n            if buckets[modality] is None:\n                buckets[modality] = [[] for _ in range(array.shape[0])]\n            coordinates = np.flatnonzero(valid)\n            if coordinates.size == 0:\n                continue\n            if coordinates.size > points_per_window:\n                coordinates = rng.choice(coordinates, points_per_window, replace=False)\n            flat = array.reshape(array.shape[0], -1)\n            for band in range(array.shape[0]):\n                buckets[modality][band].append(flat[band, coordinates])\n    result = {}\n    for modality, bands in buckets.items():\n        if bands is None:\n            raise ValueError(f"No valid {modality} values found in training windows")\n        lows, highs = [], []\n        for chunks in bands:\n            values = np.concatenate(chunks)\n            lo, hi = np.percentile(values, percentiles)\n            if not np.isfinite(lo + hi):\n                raise ValueError(f"Non-finite normalization range for {modality}")\n            if hi <= lo:\n                hi = lo + 1.0\n            lows.append(float(lo)); highs.append(float(hi))\n        result[modality] = {"low": lows, "high": highs,\n                            "percentiles": [float(percentiles[0]), float(percentiles[1])]}\n    return result\n\n\nclass RasterPatchDataset(Dataset):\n    def __init__(self, records: list[PatchRecord], ranges: dict,\n                 augment: bool = False, positive_value: int = 1,\n                 background_value: int = 0, ignore_values: Iterable[int] = ()):\n        self.records = list(records)\n        self.ranges = ranges\n        self.augment = augment\n        self.positive_value = positive_value\n        self.background_value = background_value\n        self.ignore_values = tuple(ignore_values)\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, index):\n        record = self.records[index]\n        window = Window(record.col, record.row, record.patch_size, record.patch_size)\n        s1, valid1 = _read_valid(record.s1_path, window)\n        s2, valid2 = _read_valid(record.s2_path, window)\n        target, valid_mask = _read_label(record.mask_path, window, self.positive_value,\n                                         self.background_value, self.ignore_values)\n        valid = valid1 & valid2 & valid_mask\n        for modality, array in (("s1", s1), ("s2", s2)):\n            lo = np.asarray(self.ranges[modality]["low"], dtype=np.float32)[:, None, None]\n            hi = np.asarray(self.ranges[modality]["high"], dtype=np.float32)[:, None, None]\n            array = np.clip((array - lo) / np.maximum(hi - lo, 1e-6), 0.0, 1.0)\n            array[:, ~valid] = 0.0\n            if modality == "s1":\n                s1 = array\n            else:\n                s2 = array\n        if self.augment:\n            if random.random() < 0.5:\n                s1 = np.flip(s1, axis=2); s2 = np.flip(s2, axis=2)\n                target = np.flip(target, axis=1); valid = np.flip(valid, axis=1)\n            if random.random() < 0.5:\n                s1 = np.flip(s1, axis=1); s2 = np.flip(s2, axis=1)\n                target = np.flip(target, axis=0); valid = np.flip(valid, axis=0)\n            turns = random.randrange(4)\n            if turns:\n                s1 = np.rot90(s1, turns, axes=(1, 2)); s2 = np.rot90(s2, turns, axes=(1, 2))\n                target = np.rot90(target, turns, axes=(0, 1)); valid = np.rot90(valid, turns, axes=(0, 1))\n        return (\n            torch.from_numpy(np.ascontiguousarray(s1)).float(),\n            torch.from_numpy(np.ascontiguousarray(s2)).float(),\n            torch.from_numpy(np.ascontiguousarray(target[None])).float(),\n            torch.from_numpy(np.ascontiguousarray(valid[None])).float(),\n        )\n'
(CODE_DIR / 'model.py').write_text(MODEL_SOURCE, encoding='utf-8')
(CODE_DIR / 'dataset.py').write_text(DATASET_SOURCE, encoding='utf-8')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from dataset import (canonical_tile_id, identify_modality, raster_signature,
                     validate_pair_alignment, assign_spatial_splits,
                     build_patch_records, fit_robust_ranges, RasterPatchDataset)
from model import LateFusionSegNet
print('Saved and loaded experiment modules from', CODE_DIR)

## 3. Authenticate with the Colab Secret and download the original Kaggle archive

The token is read from Colab Secrets and placed in the process environment only. This cell never prints it. The original ZIP is preserved under `raw/`; the Kaggle download is not extracted by the CLI.

In [ ]:
KAGGLE_DATASET = 'akhilchibber/deforestation-detection-dataset'
ARCHIVE_PATH = RAW_DIR / 'deforestation-detection-dataset.zip'

token = userdata.get('KAGGLE_API_TOKEN')
if not token:
    raise RuntimeError('Colab Secret KAGGLE_API_TOKEN is missing or notebook access is disabled.')
os.environ['KAGGLE_API_TOKEN'] = token
del token

if not ARCHIVE_PATH.exists():
    staging = RAW_DIR / f'download_staging_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    staging.mkdir(parents=True, exist_ok=False)
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(staging)],
        text=True, capture_output=True
    )
    if result.returncode != 0:
        raise RuntimeError('Kaggle download failed. Check the secret, Kaggle access, and Drive space.\n' + result.stderr[-3000:])
    downloaded = list(staging.glob('*.zip'))
    if len(downloaded) != 1:
        raise RuntimeError(f'Expected one Kaggle ZIP in {staging}, found {len(downloaded)}. No files were deleted.')
    with zipfile.ZipFile(downloaded[0]) as zf:
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f'Downloaded archive failed ZIP CRC verification at {bad}; staging copy is retained.')
    shutil.move(str(downloaded[0]), str(ARCHIVE_PATH))
else:
    print('Original Kaggle archive already exists; retaining it and verifying it.')

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f'Existing archive CRC failed at {bad}. It was preserved; rename it manually before retrying download.')
    entries = [item for item in zf.infolist() if not item.is_dir()]
    expected_files = len(entries)
    expected_bytes = sum(item.file_size for item in entries)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()

archive_sha256 = sha256_file(ARCHIVE_PATH)
archive_bytes = ARCHIVE_PATH.stat().st_size
print(f'Archive: {ARCHIVE_PATH.name} | {archive_bytes:,} bytes | SHA-256 {archive_sha256}')
print(f'ZIP CRC: passed | files: {expected_files:,} | extracted bytes: {expected_bytes:,} ({expected_bytes/1024**3:.2f} GiB)')
free_now = shutil.disk_usage(str(DRIVE_ROOT)).free
if free_now < expected_bytes + 2 * 1024**3:
    raise RuntimeError('Reported mounted-filesystem free space is below extracted size plus 2 GiB. Check Drive quota before extracting.')
print('Preflight complete. Also confirm sufficient actual Google Drive quota in Drive settings.')

## 4. Extract safely and verify the extracted file set

Extraction skips already complete files and writes each remaining file through a `.partial` staging path. If Colab disconnects, rerun this cell; completed files are retained. ZIP traversal paths are rejected.

In [ ]:
manifest_path = META_DIR / 'extraction_manifest.json'
existing_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if existing_manifest and existing_manifest.get('archive_sha256') != archive_sha256:
    raise RuntimeError('Extraction manifest belongs to a different archive. Existing files were preserved.')

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    infos = [item for item in zf.infolist() if not item.is_dir()]
    root = EXTRACT_DIR.resolve()
    for item in infos:
        destination = (root / item.filename).resolve()
        if root not in destination.parents:
            raise RuntimeError(f'Unsafe archive member path: {item.filename}')
    for number, item in enumerate(infos, 1):
        destination = root / item.filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size == item.file_size:
            continue
        partial = destination.with_name(destination.name + '.partial')
        with zf.open(item, 'r') as source, open(partial, 'wb') as target:
            shutil.copyfileobj(source, target, length=8 * 1024 * 1024)
        if partial.stat().st_size != item.file_size:
            raise IOError(f'Extracted size mismatch: {item.filename}')
        os.replace(partial, destination)
        if number % 500 == 0:
            print(f'Extracted {number:,}/{len(infos):,} files')

actual_files = [p for p in EXTRACT_DIR.rglob('*') if p.is_file() and not p.name.endswith('.partial')]
actual_bytes = sum(p.stat().st_size for p in actual_files)
if len(actual_files) != expected_files or actual_bytes != expected_bytes:
    raise RuntimeError(f'Extraction verification mismatch: expected {expected_files} files/{expected_bytes} bytes; got {len(actual_files)}/{actual_bytes}.')
manifest = {
    'dataset': KAGGLE_DATASET, 'archive': str(ARCHIVE_PATH), 'archive_bytes': archive_bytes,
    'archive_sha256': archive_sha256, 'zip_crc': 'passed', 'extracted_files': len(actual_files),
    'extracted_bytes': actual_bytes, 'verified_utc': datetime.now(timezone.utc).isoformat()
}
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## 5. Inspect folder layout, modality counts, TIFF metadata, masks, and pair matches

The author’s cloud-free research notebooks use separate `SENTINEL1/IMAGE_16_GRID`, `SENTINEL2/IMAGE_16_GRID`, and `MASK_16_GRID` files; grid names such as `RASTER_12.tif` are matched by ID. This cell discovers the extracted layout rather than assuming a fixed outer folder name. It reports cloudy files too, but the primary experiment selects only author-provided cloud-free inputs.

In [ ]:
all_tiffs = sorted([*EXTRACT_DIR.rglob('*.tif'), *EXTRACT_DIR.rglob('*.tiff')])
inventory = []
for path in all_tiffs:
    modality, domain = identify_modality(path)
    inventory.append({'path': str(path), 'relative_path': str(path.relative_to(EXTRACT_DIR)),
                      'modality': modality or 'unclassified', 'domain': domain,
                      'tile_id': canonical_tile_id(path), 'bytes': path.stat().st_size})
inventory_df = pd.DataFrame(inventory)
if inventory_df.empty:
    raise RuntimeError(f'No GeoTIFFs found below {EXTRACT_DIR}. Inspect the extracted file listing.')
display(inventory_df.groupby(['domain', 'modality']).agg(files=('path','count'), bytes=('bytes','sum')).reset_index())
inventory_df.to_csv(META_DIR / 'tiff_inventory.csv', index=False)

classified = {}
mask_paths = {}
for item in inventory:
    tile = item['tile_id']
    if item['modality'] == 'mask':
        mask_paths.setdefault(tile, []).append(item['path'])
    elif item['modality'] in ('s1', 's2'):
        key = (item['domain'], item['modality'], tile)
        if key in classified:
            raise RuntimeError(f'Duplicate image key {key}: {classified[key]} and {item["path"]}')
        classified[key] = item['path']

domains = sorted({key[0] for key in classified})
print('Detected image domains:', domains)
print('Mask grid files:', len(mask_paths))
for domain in domains:
    print(domain, 'S1=', sum(k[0] == domain and k[1] == 's1' for k in classified),
          'S2=', sum(k[0] == domain and k[1] == 's2' for k in classified))

def choose_mask(tile):
    candidates = mask_paths.get(tile, [])
    if len(candidates) == 1:
        return candidates[0]
    preferred = [p for p in candidates if 'mask_16_grid' in p.casefold()]
    if len(preferred) == 1:
        return preferred[0]
    if not candidates:
        return None
    raise RuntimeError(f'Multiple ambiguous masks for {tile}: {candidates}')

def assemble_domain(domain):
    tiles = sorted({key[2] for key in classified if key[0] == domain},
                   key=lambda value: int(value.split('_')[-1]) if value.split('_')[-1].isdigit() else value)
    samples, unmatched = [], []
    for tile in tiles:
        s1 = classified.get((domain, 's1', tile)); s2 = classified.get((domain, 's2', tile))
        mask = choose_mask(tile)
        if not (s1 and s2 and mask):
            unmatched.append({'domain': domain, 'tile_id': tile, 'has_s1': bool(s1),
                              'has_s2': bool(s2), 'has_mask': bool(mask)})
            continue
        samples.append({'tile_id': tile, 'domain': domain, 's1_path': s1,
                        's2_path': s2, 'mask_path': mask})
    return samples, unmatched

cloudfree_samples, cloudfree_unmatched = assemble_domain('cloud_free')
cloudy_samples, cloudy_unmatched = assemble_domain('cloudy')
print(f'Cloud-free complete S1/S2/mask pairs: {len(cloudfree_samples)}')
print(f'Cloudy complete S1/S2/mask pairs (inspection only): {len(cloudy_samples)}')
print('Cloud-free unmatched IDs:', cloudfree_unmatched)
print('Cloudy unmatched IDs:', cloudy_unmatched)
if not cloudfree_samples:
    raise RuntimeError('No complete cloud-free S1/S2/mask triples were discovered. Review inventory before proceeding.')

profile_rows, alignment_rows = [], []
for sample in cloudfree_samples + cloudy_samples:
    for role in ('s1_path', 's2_path', 'mask_path'):
        signature = raster_signature(sample[role])
        profile_rows.append({'domain': sample['domain'], 'tile_id': sample['tile_id'],
                             'role': role, **{k: signature[k] for k in ('width','height','count','dtype','crs','transform','bounds','nodata','descriptions')}})
    okay, message = validate_pair_alignment(sample['s1_path'], sample['s2_path'], sample['mask_path'])
    alignment_rows.append({'domain': sample['domain'], 'tile_id': sample['tile_id'],
                           'aligned': okay, 'result': message})
profiles_df = pd.DataFrame(profile_rows)
alignment_df = pd.DataFrame(alignment_rows)
display(profiles_df.head(30))
display(alignment_df.groupby(['domain','aligned']).size().rename('tiles').reset_index())
profiles_df.to_json(META_DIR / 'raster_profiles.json', orient='records', indent=2)
alignment_df.to_csv(META_DIR / 'alignment_checks.csv', index=False)

failed_alignment = alignment_df[~alignment_df.aligned]
if len(failed_alignment):
    print('Tiles excluded due to failed CRS/transform/dimension checks:')
    display(failed_alignment)
cloudfree_samples = [sample for sample in cloudfree_samples
                     if bool(alignment_df.query('domain == "cloud_free" and tile_id == @sample["tile_id"]').aligned.iloc[0])]
if not cloudfree_samples:
    raise RuntimeError('All cloud-free triples failed alignment checks. No resampling was performed.')

s1_counts = {raster_signature(s['s1_path'])['count'] for s in cloudfree_samples}
s2_counts = {raster_signature(s['s2_path'])['count'] for s in cloudfree_samples}
print('Verified cloud-free band counts: Sentinel-1', s1_counts, '| Sentinel-2', s2_counts)
if len(s1_counts) != 1 or len(s2_counts) != 1:
    raise RuntimeError('Band counts differ across cloud-free tiles. Resolve this before building batches.')
S1_CHANNELS, S2_CHANNELS = next(iter(s1_counts)), next(iter(s2_counts))
print('Research notebook expects 2 SAR and 4 optical bands; actual bands:', S1_CHANNELS, S2_CHANNELS)

mask_value_counts = {}
for sample in cloudfree_samples:
    with rasterio.open(sample['mask_path']) as src:
        values, counts = np.unique(src.read(1), return_counts=True)
    for value, count in zip(values.tolist(), counts.tolist()):
        mask_value_counts[int(value)] = mask_value_counts.get(int(value), 0) + int(count)
mask_nodata_values = {rasterio.open(s['mask_path']).nodata for s in cloudfree_samples}
mask_nodata_values.discard(None)
print('PRODES mask raster values / counts:', mask_value_counts)
print('Mask nodata metadata values:', mask_nodata_values)

# The author’s training notebook treats source value 1 as deforestation and
# every valid non-1 pixel as non-deforestation. Confirm this against observed values.
POSITIVE_VALUE, BACKGROUND_VALUE = 1, 0
if mask_nodata_values & {POSITIVE_VALUE, BACKGROUND_VALUE}:
    raise RuntimeError(f'Mask nodata collides with a label class: {mask_nodata_values & {POSITIVE_VALUE, BACKGROUND_VALUE}}. Confirm the source encoding before proceeding.')
IGNORE_VALUES = set(mask_value_counts) - {POSITIVE_VALUE, BACKGROUND_VALUE}
unexplained = IGNORE_VALUES - mask_nodata_values
if unexplained:
    raise RuntimeError(f'Unexpected mask classes {unexplained}. Confirm their meaning before assigning labels.')
if POSITIVE_VALUE not in mask_value_counts or BACKGROUND_VALUE not in mask_value_counts:
    raise RuntimeError('Expected mask values 1=deforestation and 0=background from the original notebook were not found. Review and update the mapping explicitly.')
valid_mask_pixels = sum(count for value, count in mask_value_counts.items()
                        if value in (POSITIVE_VALUE, BACKGROUND_VALUE))
positive_mask_pixels = mask_value_counts[POSITIVE_VALUE]
print(f'Verified mapped mask pixels: {valid_mask_pixels:,}; positive: {positive_mask_pixels:,} ({positive_mask_pixels/max(valid_mask_pixels,1):.4%})')

dataset_report = {
    'dataset': KAGGLE_DATASET, 'archive_path': str(ARCHIVE_PATH),
    'archive_bytes': archive_bytes, 'archive_sha256': archive_sha256,
    'extracted_files': len(actual_files), 'extracted_bytes': actual_bytes,
    'tiff_files': len(all_tiffs), 'domains': domains,
    'cloud_free_s1_s2_mask_pairs': len(cloudfree_samples),
    'cloudy_s1_s2_mask_pairs_inspected_not_used': len(cloudy_samples),
    'cloud_free_unmatched': cloudfree_unmatched,
    'mask_value_counts': mask_value_counts, 'mask_nodata_values': list(mask_nodata_values),
    'positive_mask_pixels': positive_mask_pixels,
    'alignment_failures': failed_alignment.to_dict(orient='records'),
    's1_channels': S1_CHANNELS, 's2_channels': S2_CHANNELS,
}
(META_DIR / 'dataset_report.json').write_text(json.dumps(dataset_report, indent=2, default=str))
print('Saved actual inventory report:', META_DIR / 'dataset_report.json')

## 6. Visual inspection before preprocessing

Inspect optical bands individually because their exact band order is read from each TIFF and is not guessed as RGB. The overlay uses the confirmed binary mask. Cloudy examples are displayed separately and are not added to training.

In [ ]:
def percentile_display(array):
    array = np.asarray(array, dtype=np.float32)
    shown = np.zeros_like(array, dtype=np.float32)
    for band in range(array.shape[0]):
        plane = array[band]
        finite = np.isfinite(plane)
        if not finite.any():
            continue
        lo, hi = np.percentile(plane[finite], (2, 98))
        shown[band] = np.clip((plane - lo) / max(hi - lo, 1e-6), 0, 1)
    return shown

from rasterio.windows import Window

def preview_pair(sample, title):
    with rasterio.open(sample['s1_path']) as src:
        window = Window(0, 0, min(512, src.width), min(512, src.height))
        s1 = src.read(window=window).astype(np.float32)
    with rasterio.open(sample['s2_path']) as src:
        s2 = src.read(window=window).astype(np.float32)
    with rasterio.open(sample['mask_path']) as src:
        mask = src.read(1, window=window)
    s1v, s2v = percentile_display(s1), percentile_display(s2)
    fig, axes = plt.subplots(1, min(S2_CHANNELS, 6) + 2, figsize=(4*(min(S2_CHANNELS,6)+2), 4))
    axes[0].imshow(s1v[0], cmap='gray'); axes[0].set_title('S1 band 1')
    for band in range(min(S2_CHANNELS, 6)):
        axes[band+1].imshow(s2v[band], cmap='gray')
        axes[band+1].set_title(f'S2 band {band+1}')
    overlay_axis = axes[-1]
    overlay_axis.imshow(np.moveaxis(s2v[:min(3,S2_CHANNELS)],0,-1))
    positive = (mask == POSITIVE_VALUE)
    overlay_axis.imshow(np.ma.masked_where(~positive, positive), cmap='autumn', alpha=.55, vmin=0, vmax=1)
    overlay_axis.set_title('S2 first bands + PRODES mask')
    for axis in axes: axis.axis('off')
    fig.suptitle(f'{title}: {sample["tile_id"]}')
    fig.tight_layout()
    return fig

for sample in cloudfree_samples[:3]:
    fig = preview_pair(sample, 'CLOUD-FREE')
    fig.savefig(VIS_DIR / f'preview_cloudfree_{sample["tile_id"]}.png', dpi=130)
    plt.show()

cloudy_by_id = {sample['tile_id']: sample for sample in cloudy_samples}
common_domain_ids = sorted(set(cloudy_by_id) & {sample['tile_id'] for sample in cloudfree_samples})
if common_domain_ids:
    chosen = common_domain_ids[0]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, sample, label in ((axes[0], next(s for s in cloudfree_samples if s['tile_id']==chosen), 'cloud-free'),
                                (axes[1], cloudy_by_id[chosen], 'cloudy (inspection only)')):
        with rasterio.open(sample['s2_path']) as src: image = src.read(window=Window(0, 0, min(512, src.width), min(512, src.height))).astype(np.float32)
        shown = percentile_display(image)
        axis.imshow(np.moveaxis(shown[:min(3,S2_CHANNELS)],0,-1)); axis.set_title(label); axis.axis('off')
    fig.suptitle(f'Cloud-condition comparison: {chosen}')
    fig.tight_layout(); fig.savefig(VIS_DIR / f'cloud_comparison_{chosen}.png', dpi=130); plt.show()
else:
    print('No same-ID cloudy/cloud-free pair was available for side-by-side inspection.')

## 7. Make spatial splits before creating patches

Aligned raster tiles are grouped into adjacent 2×2 geographic blocks from their geotransforms. Blocks are assigned as indivisible units before window extraction. Actual ratios can differ from 70/15/15 when only a few blocks are available; the tile/block map is saved for review.

In [ ]:
tile_signatures = []
for sample in cloudfree_samples:
    sig = raster_signature(sample['s1_path'])
    tile_signatures.append({'tile_id': sample['tile_id'], 'crs': sig['crs'], 'bounds': sig['bounds']})
tile_splits, tile_blocks = assign_spatial_splits(tile_signatures, seed=42)
for sample in cloudfree_samples:
    sample['split'] = tile_splits[sample['tile_id']]
    sample['spatial_block'] = tile_blocks[sample['tile_id']]

split_table = pd.DataFrame([{'tile_id': s['tile_id'], 'split': s['split'],
                             'spatial_block': s['spatial_block'],
                             'bounds': raster_signature(s['s1_path'])['bounds']}
                            for s in cloudfree_samples])
display(split_table.groupby(['split','spatial_block']).size().rename('tiles').reset_index())
split_table.to_csv(META_DIR / 'spatial_tile_splits.csv', index=False)
if set(split_table.split) != {'train','val','test'}:
    raise RuntimeError(f'Spatial split is incomplete: {split_table.split.value_counts().to_dict()}')

fig, ax = plt.subplots(figsize=(8,7))
colors = {'train':'#2e8b57','val':'#e0a526','test':'#d9534f'}
for sample in cloudfree_samples:
    xmin,ymin,xmax,ymax = raster_signature(sample['s1_path'])['bounds']
    rect=plt.Rectangle((xmin,ymin),xmax-xmin,ymax-ymin,fill=False,linewidth=2,color=colors[sample['split']])
    ax.add_patch(rect); ax.text((xmin+xmax)/2,(ymin+ymax)/2,sample['tile_id'],ha='center',va='center',fontsize=8)
ax.autoscale(); ax.set_aspect('equal', adjustable='datalim'); ax.set_title('Whole-tile spatial splits (projected coordinates)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); plt.show()

## 8. Index 256×256 aligned patches and fit normalization on train only

Patches do not cross source raster boundaries and use a non-overlapping stride of 256. Invalid pixels and any mask value outside the verified 0/1 mapping are ignored. No interpolation or label resampling occurs.

In [ ]:
PATCH_SIZE = 256
MIN_VALID_FRACTION = 0.90
samples_for_index = [{k: s[k] for k in ('tile_id','s1_path','s2_path','mask_path')} for s in cloudfree_samples]
patch_records = build_patch_records(
    samples_for_index, tile_splits, patch_size=PATCH_SIZE, stride=PATCH_SIZE,
    positive_value=POSITIVE_VALUE, background_value=BACKGROUND_VALUE,
    ignore_values=IGNORE_VALUES, min_valid_fraction=MIN_VALID_FRACTION
)
if not patch_records:
    raise RuntimeError('No patches passed paired-image/mask validity checks.')
patch_df = pd.DataFrame([record.to_dict() for record in patch_records])
patch_df.to_csv(PATCH_DIR / 'patch_index.csv', index=False)
display(patch_df.groupby('split').agg(patches=('tile_id','size'), tiles=('tile_id','nunique'),
                                       mean_valid=('valid_fraction','mean'),
                                       mean_positive=('positive_fraction','mean')).reset_index())
positive_by_split = patch_df.groupby('split').apply(lambda group: int((group.positive_fraction > 0).sum()), include_groups=False)
print('Patches containing any positive pixels:', positive_by_split.to_dict())
if positive_by_split.get('train',0) == 0:
    raise RuntimeError('Training split has no positive patches; review geographic blocks and source labels.')
if positive_by_split.get('test',0) == 0:
    warnings.warn('Test split has no positive patches. Report metrics as limited; do not infer deforestation recall.')

normalization = fit_robust_ranges(
    patch_records, percentiles=(1,99), max_windows=800, points_per_window=512,
    seed=42, positive_value=POSITIVE_VALUE, background_value=BACKGROUND_VALUE,
    ignore_values=IGNORE_VALUES
)
(META_DIR / 'normalization_ranges.json').write_text(json.dumps(normalization, indent=2))
print('Train-only robust percentile ranges:', json.dumps(normalization, indent=2))

train_records = [r for r in patch_records if r.split == 'train']
train_positive = sum(r.positive_fraction * r.valid_fraction * PATCH_SIZE**2 for r in train_records)
train_valid = sum(r.valid_fraction * PATCH_SIZE**2 for r in train_records)
train_negative = max(train_valid - train_positive, 0)
POS_WEIGHT = float(np.clip(train_negative / max(train_positive, 1), 1.0, 20.0))
print(f'Train valid positive-pixel estimate={train_positive:,.0f}; negative={train_negative:,.0f}; BCE pos_weight={POS_WEIGHT:.2f}')

## 9. Create loaders and check tensor shapes

Train-only geometric augmentation uses paired flips and 90° rotations. The weighted sampler modestly increases the chance of seeing positive-containing patches; validation and test keep their natural spatial distribution.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_ds = RasterPatchDataset([r for r in patch_records if r.split=='train'], normalization,
                              augment=True, positive_value=POSITIVE_VALUE,
                              background_value=BACKGROUND_VALUE, ignore_values=IGNORE_VALUES)
val_ds = RasterPatchDataset([r for r in patch_records if r.split=='val'], normalization,
                            augment=False, positive_value=POSITIVE_VALUE,
                            background_value=BACKGROUND_VALUE, ignore_values=IGNORE_VALUES)
test_ds = RasterPatchDataset([r for r in patch_records if r.split=='test'], normalization,
                             augment=False, positive_value=POSITIVE_VALUE,
                             background_value=BACKGROUND_VALUE, ignore_values=IGNORE_VALUES)
if not len(train_ds) or not len(val_ds) or not len(test_ds):
    raise RuntimeError('At least one spatial split has zero valid patches. Stop and review the data/split map.')

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training; this notebook will not train on your laptop.')
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH_SIZE = 4 if vram_gib >= 24 else (2 if vram_gib >= 10 else 1)
NUM_WORKERS = 2
USE_AMP = True
print(f'GPU VRAM {vram_gib:.1f} GiB -> batch size {BATCH_SIZE}; AMP={USE_AMP}')

sampler_weights = [3.0 if r.positive_fraction >= 0.005 else 1.0 for r in train_ds.records]
sampler = WeightedRandomSampler(torch.as_tensor(sampler_weights,dtype=torch.double),
                                 num_samples=len(sampler_weights), replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
sample_batch = next(iter(train_loader))
print('Batch shapes S1, S2, target, valid:', [tuple(t.shape) for t in sample_batch])
assert sample_batch[0].shape[1] == S1_CHANNELS and sample_batch[1].shape[1] == S2_CHANNELS

## 10. Build the separate late-fusion SegNet

Each modality has its own VGG-style encoder, max-pooling indices, and SegNet decoder. Decoded features are concatenated and passed through a small fusion head. The model returns logits; it does not concatenate Sentinel-1 and Sentinel-2 into a temporal input.

In [ ]:
device = torch.device('cuda')
model = LateFusionSegNet(s1_channels=S1_CHANNELS, s2_channels=S2_CHANNELS, base_channels=32).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Model parameters: {parameter_count:,} ({parameter_count*4/1024**2:.1f} MiB FP32 weights)')
with torch.no_grad():
    check = model(sample_batch[0].to(device), sample_batch[1].to(device))
print('Logit shape:', tuple(check.shape))
assert check.shape == sample_batch[2].shape

## 11. Train with masked BCE + Dice, validation monitoring, and Drive checkpoints

Every completed epoch saves a resumable checkpoint to Drive. `best_model.pt` is selected using validation IoU. To resume after a disconnect, rerun this cell with `RESUME=True`; the notebook never starts a second unrelated experiment silently.

In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

MAX_EPOCHS = 50
PATIENCE = 8
RESUME = True
RUN_DIR = RESULT_DIR / 'late_fusion_segnet_v1'
RUN_DIR.mkdir(parents=True, exist_ok=True)
LAST_PATH = MODEL_DIR / 'late_fusion_segnet_last.pt'
BEST_PATH = MODEL_DIR / 'late_fusion_segnet_best.pt'
HISTORY_PATH = RUN_DIR / 'history.csv'

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = GradScaler(enabled=USE_AMP)
start_epoch, best_val_iou, bad_epochs, history = 0, -1.0, 0, []
if RESUME and LAST_PATH.exists():
    state = torch.load(LAST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    if state.get('scaler'): scaler.load_state_dict(state['scaler'])
    start_epoch = state['epoch'] + 1
    best_val_iou = state['best_val_iou']; bad_epochs = state['bad_epochs']; history = state['history']
    print('Resuming after epoch', start_epoch)

pos_weight_tensor = torch.tensor([POS_WEIGHT], device=device)

def masked_loss(logits, target, valid):
    elementwise = F.binary_cross_entropy_with_logits(logits, target,
                                                      pos_weight=pos_weight_tensor,
                                                      reduction='none')
    bce = (elementwise * valid).sum() / valid.sum().clamp_min(1)
    probability = torch.sigmoid(logits) * valid
    truth = target * valid
    dice = (2 * (probability * truth).sum() + 1) / (probability.sum() + truth.sum() + 1)
    return bce + (1 - dice)

@torch.no_grad()
def run_validation(loader):
    model.eval(); total_loss=0.0; batches=0
    tn=fp=fn=tp=0
    for s1,s2,target,valid in loader:
        s1,s2,target,valid=[x.to(device,non_blocking=True) for x in (s1,s2,target,valid)]
        with autocast(enabled=USE_AMP): logits=model(s1,s2)
        loss=masked_loss(logits.float(),target,valid)
        prediction=(torch.sigmoid(logits.float()) >= 0.5)
        truth=target >= 0.5; mask=valid >= 0.5
        tp += int((prediction & truth & mask).sum()); fp += int((prediction & ~truth & mask).sum())
        fn += int((~prediction & truth & mask).sum()); tn += int((~prediction & ~truth & mask).sum())
        total_loss += float(loss); batches += 1
    precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)
    dice=2*tp/max(2*tp+fp+fn,1); iou=tp/max(tp+fp+fn,1)
    accuracy=(tp+tn)/max(tp+tn+fp+fn,1)
    return {'loss':total_loss/max(batches,1),'iou':iou,'dice_f1':dice,
            'precision':precision,'recall':recall,'accuracy':accuracy,
            'tn':tn,'fp':fp,'fn':fn,'tp':tp}

def save_checkpoint(path, epoch, best, bad, history_rows):
    payload={'epoch':epoch,'best_val_iou':best,'bad_epochs':bad,'history':history_rows,
             'model':model.state_dict(),'optimizer':optimizer.state_dict(),
             'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
             'config':{'s1_channels':S1_CHANNELS,'s2_channels':S2_CHANNELS,
                       'patch_size':PATCH_SIZE,'base_channels':32,'pos_weight':POS_WEIGHT}}
    temporary=path.with_suffix(path.suffix+'.partial')
    torch.save(payload,temporary); os.replace(temporary,path)

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train(); running=0.0; seen=0
    for s1,s2,target,valid in train_loader:
        s1,s2,target,valid=[x.to(device,non_blocking=True) for x in (s1,s2,target,valid)]
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP): logits=model(s1,s2); loss=masked_loss(logits,target,valid)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running += float(loss.detach()); seen += 1
    val=run_validation(val_loader)
    scheduler.step(val['iou'])
    row={'epoch':epoch+1,'train_loss':running/max(seen,1),
         'val_loss':val['loss'],'val_iou':val['iou'],'val_dice_f1':val['dice_f1'],
         'val_precision':val['precision'],'val_recall':val['recall'],
         'val_accuracy':val['accuracy'],'lr':optimizer.param_groups[0]['lr']}
    history.append(row)
    if val['iou'] > best_val_iou:
        best_val_iou=val['iou']; bad_epochs=0
        best_tmp=BEST_PATH.with_suffix('.pt.partial')
        torch.save({'model':model.state_dict(),'epoch':epoch+1,'val_metrics':val,
                    'config':{'s1_channels':S1_CHANNELS,'s2_channels':S2_CHANNELS,
                              'patch_size':PATCH_SIZE,'base_channels':32}},best_tmp)
        os.replace(best_tmp,BEST_PATH)
    else:
        bad_epochs += 1
    save_checkpoint(LAST_PATH,epoch,best_val_iou,bad_epochs,history)
    pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
    print(f"Epoch {epoch+1:03d}: train_loss={row['train_loss']:.4f} val_loss={val['loss']:.4f} "
          f"val_IoU={val['iou']:.4f} val_Dice={val['dice_f1']:.4f} patience={bad_epochs}/{PATIENCE}")
    if bad_epochs >= PATIENCE:
        print('Early stopping on validation IoU.')
        break

if not BEST_PATH.exists():
    raise RuntimeError('No completed training epoch produced a best checkpoint.')
print('Best model:', BEST_PATH)
print('Resume checkpoint:', LAST_PATH)

## 12. Evaluate once on the held-out geographic test split

The test set is not used for checkpoint selection or threshold tuning. Threshold is fixed at 0.5. Metrics exclude invalid and unknown-label pixels.

In [ ]:
best_state=torch.load(BEST_PATH,map_location=device,weights_only=False)
model.load_state_dict(best_state['model'])
test_metrics=run_validation(test_loader)
print(json.dumps(test_metrics,indent=2))
confusion=np.array([[test_metrics['tn'],test_metrics['fp']],
                    [test_metrics['fn'],test_metrics['tp']]],dtype=np.int64)
pd.DataFrame(confusion,index=['actual_0','actual_1'],columns=['pred_0','pred_1']).to_csv(RESULT_DIR/'confusion_matrix.csv')
(RESULT_DIR/'test_metrics.json').write_text(json.dumps(test_metrics,indent=2))
print('Confusion matrix [[TN,FP],[FN,TP]]:\n',confusion)

## 13. Save curves and held-out mask comparisons

In [ ]:
history_df=pd.read_csv(HISTORY_PATH)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history_df.epoch,history_df.train_loss,label='train')
axes[0].plot(history_df.epoch,history_df.val_loss,label='validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history_df.epoch,history_df.val_iou,label='validation IoU')
axes[1].plot(history_df.epoch,history_df.val_dice_f1,label='validation Dice/F1')
axes[1].set_title('Validation metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()
fig.tight_layout(); fig.savefig(VIS_DIR/'training_curves.png',dpi=160); plt.show()

def show_test_predictions(max_items=6):
    model.eval(); shown=0
    for index in range(len(test_ds)):
        s1,s2,target,valid=test_ds[index]
        with torch.no_grad(), autocast(enabled=USE_AMP):
            probability=torch.sigmoid(model(s1[None].to(device),s2[None].to(device))).cpu().numpy()[0,0]
        truth=target.numpy()[0]; mask=valid.numpy()[0] > .5
        s2_display=percentile_display(s2.numpy())
        first3=np.moveaxis(s2_display[:min(3,S2_CHANNELS)],0,-1)
        predicted=(probability >= .5) & mask
        overlay=first3.copy()
        overlay[predicted]=.55*overlay[predicted]+.45*np.array([1.0,.1,.1])
        fig,axes=plt.subplots(1,5,figsize=(16,3.5))
        axes[0].imshow(s1.numpy()[0],cmap='gray'); axes[0].set_title('S1 band 1')
        axes[1].imshow(first3); axes[1].set_title('S2 first 3 bands')
        axes[2].imshow(np.ma.masked_where(~mask,truth),cmap='gray',vmin=0,vmax=1); axes[2].set_title('PRODES truth')
        axes[3].imshow(np.ma.masked_where(~mask,probability),cmap='magma',vmin=0,vmax=1); axes[3].set_title('Probability')
        axes[4].imshow(overlay); axes[4].set_title('Predicted loss overlay')
        for axis in axes: axis.axis('off')
        fig.tight_layout(); fig.savefig(VIS_DIR/f'test_prediction_{shown+1:02d}.png',dpi=140); plt.show()
        shown += 1
        if shown >= max_items: break
show_test_predictions()

## 14. Write an experiment report from measured outputs

This cell writes actual dataset counts, split sizes, runtime configuration, metrics, and artifact paths to Drive. It creates no placeholder scores.

In [ ]:
split_sizes=patch_df.groupby('split').size().to_dict()
split_tile_counts=split_table.groupby('split').tile_id.nunique().to_dict()
gpu_name=torch.cuda.get_device_name(0)
report=f"""# Kaggle Amazon Deforestation Experiment Report

**Run status:** TRAINED; report generated {datetime.now(timezone.utc).isoformat()}.

## Dataset verification
- Kaggle source: `{KAGGLE_DATASET}`
- Original archive: `{ARCHIVE_PATH}` ({archive_bytes:,} bytes; SHA-256 `{archive_sha256}`)
- Extracted files/bytes: {len(actual_files):,} / {actual_bytes:,}
- TIFF files: {len(all_tiffs):,}
- Cloud-free paired aligned tiles: {len(cloudfree_samples):,}
- Cloudy paired tiles inspected, excluded from training: {len(cloudy_samples):,}
- Verified bands: Sentinel-1={S1_CHANNELS}, Sentinel-2={S2_CHANNELS}
- Mask raw counts: `{json.dumps(mask_value_counts)}`; positive source value 1, background 0; unknown pixels ignored.
- Alignment failures excluded: {len(failed_alignment)}
- Geographic tiles by split: `{json.dumps(split_tile_counts)}`
- Valid non-overlap patches by split: `{json.dumps(split_sizes)}`
- Train-only per-band robust ranges: `{json.dumps(normalization)}`

## Model/training
- Architecture: dual-stream SegNet with late feature fusion; modality-specific branches.
- GPU: {gpu_name} ({vram_gib:.1f} GiB VRAM); batch size {BATCH_SIZE}; mixed precision {USE_AMP}.
- Best checkpoint: `{BEST_PATH}` (epoch {best_state.get('epoch')}).
- Resume checkpoint: `{LAST_PATH}`.
- Epochs completed: {len(history_df)}.

## Held-out geographic test metrics (threshold 0.5)
- IoU: {test_metrics['iou']:.6f}
- Dice/F1: {test_metrics['dice_f1']:.6f}
- Precision: {test_metrics['precision']:.6f}
- Recall: {test_metrics['recall']:.6f}
- Accuracy: {test_metrics['accuracy']:.6f}
- Confusion matrix [[TN, FP], [FN, TP]]: `{confusion.tolist()}`

## Artifacts
- History: `{HISTORY_PATH}`
- Curves: `{VIS_DIR / 'training_curves.png'}`
- Predictions/overlays: `{VIS_DIR}`
- Confusion matrix: `{RESULT_DIR / 'confusion_matrix.csv'}`

## Limitations
- The cloudy domain was inspected but excluded from training/evaluation; these results do not establish cloudy-scene robustness.
- Labels inherit PRODES source limitations. Geospatial alignment checks do not establish label correctness beyond the source annotation.
- Realized geographic split ratios may differ from 70/15/15 because whole spatial blocks were kept intact.
"""
(RESULT_DIR/'EXPERIMENT_REPORT.md').write_text(report,encoding='utf-8')
print(report)
print('Saved measured report to', RESULT_DIR/'EXPERIMENT_REPORT.md')

## 15. Artifacts and recovery

All downloaded, extracted, processed, model, metric, report, and visualization files are under `/content/drive/MyDrive/Deforestation_Research/Kaggle_Amazon/`. Keep the original ZIP. For a later Colab session, mount Drive again and rerun setup/inventory cells; the archive and extraction manifest prevent redownloading/re-extracting verified files. Training resumes from the latest complete epoch checkpoint.